# ViralScan Getting Started (Offline-Safe)

This notebook is a practical walkthrough for running ViralScan end-to-end while staying robust on HPC or restricted-network environments.

What you will do:
1. Review installation commands.
2. Build a combined host + viral reference.
3. Run the ViralScan quantification pipeline.
4. Inspect `viral_summary.tsv` and `report.html`.
5. Check UMAP outputs if enabled.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

RUN_COMMANDS = False
REPO = Path.cwd()


def run(cmd: str) -> int:
    print(f"$ {cmd}")
    if not RUN_COMMANDS:
        print("RUN_COMMANDS=False -> dry-run only")
        return 0
    return subprocess.run(cmd, shell=True, check=False).returncode

print(f"Repository: {REPO}")

## 1) Installation

Use one of the following (terminal commands):

```bash
python -m pip install -e .
```

```bash
conda env create -f environment.yml
conda activate viralscan
python -m pip install -e .
```

Quick smoke check:

```bash
PYTHONPATH=src python -m viralscan.menu --help
```

## 2) Build Reference

Example command (host human + one viral accession):

```bash
viralscan build-ref \
  --host human \
  --virus-accessions NC_045512.2 \
  --no-kb-ref \
  --output tutorial_ref
```

Use `--no-kb-ref` for a fast setup check. Remove it when you want to build the kallisto index.

In [ ]:
build_ref_cmd = " ".join([
    "viralscan build-ref",
    "--host human",
    "--virus-accessions NC_045512.2",
    "--no-kb-ref",
    "--output tutorial_ref",
])
run(build_ref_cmd)

## 3) Run ViralScan

Replace the FASTQ and reference paths with your files.

```bash
viralscan \
  -o tutorial_run \
  -s1 /path/to/sample_R1.fastq.gz \
  -s2 /path/to/sample_R2.fastq.gz \
  -i /path/to/index.idx \
  -t /path/to/t2g.txt \
  --visual \
  --umap
```

Tip: add `--cell-types /path/to/barcode_celltype.csv` to enable per-cell-type enrichment in outputs.

In [ ]:
run_cmd = " ".join([
    "viralscan",
    "-o tutorial_run",
    "-s1 /path/to/sample_R1.fastq.gz",
    "-s2 /path/to/sample_R2.fastq.gz",
    "-i /path/to/index.idx",
    "-t /path/to/t2g.txt",
    "--visual",
    "--umap",
])
run(run_cmd)

## 4) Inspect Core Outputs

After a run completes, review:
- `tutorial_run/results/viral_summary.tsv`
- `tutorial_run/results/per_cell_viral.tsv`
- `tutorial_run/results/cell_type_enrichment.tsv` (if `--cell-types` used)
- `tutorial_run/report.html`

In [ ]:
out = Path("tutorial_run")
summary_tsv = out / "results" / "viral_summary.tsv"
report_html = out / "report.html"

print(f"summary exists: {summary_tsv.exists()} -> {summary_tsv}")
print(f"report exists:  {report_html.exists()} -> {report_html}")

if summary_tsv.exists():
    display(pd.read_csv(summary_tsv, sep="\t").head(20))

## 5) UMAP Outputs

If `--umap` was enabled, ViralScan writes interactive HTML plots:
- `tutorial_run/plots/umap_binary.html`
- `tutorial_run/plots/umap_continuous.html`

In [ ]:
from IPython.display import IFrame

umap_binary = out / "plots" / "umap_binary.html"
umap_cont = out / "plots" / "umap_continuous.html"

print(f"binary umap exists:     {umap_binary.exists()} -> {umap_binary}")
print(f"continuous umap exists: {umap_cont.exists()} -> {umap_cont}")

if umap_binary.exists():
    IFrame(str(umap_binary), width=900, height=600)